# Semana 8 · Organización y Almacenamiento
## Proyecto: Sistema de alerta temprana — arañita roja, Fundo Loma Verde S.A.C.

### Entrada
`clima_validado_semana7.csv` y `fitosanitario_validado_semana7.csv`, generados en la Semana 7.

### Producto
Repositorio de datos organizado: base SQLite con esquema relacional, diccionario de datos, manifiesto de versión, exportación controlada, respaldo y prueba de recuperación.

### Diseño del modelo (adaptado al patrón del caso guía, no a sus reglas literales)
A diferencia del caso guía (una sola tabla de eventos), aquí tenemos **dos fuentes a distinta granularidad**:
- `clima_diario`: una fila por día (grano: fecha) — la estación es única para todo el fundo.
- `evaluacion_fitosanitaria`: una fila por evaluación de campo (grano: fecha + módulo + válvula + turno).

Se conectan por `fecha` (clave foránea de `evaluacion_fitosanitaria` hacia `clima_diario`), igual que un "evento" se conecta a sus dimensiones en el caso guía.

`válvula` **no** se modela como dimensión aparte: los números de válvula se repiten entre módulos (no son únicos globalmente), así que solo tienen sentido junto a su módulo. Se deja como atributo dentro de `evaluacion_fitosanitaria`, no como tabla propia.

## BLOQUE 1 · Librerías

In [ ]:
import pandas as pd
import numpy as np
import sqlite3
import os
import shutil
import json
from pathlib import Path
from datetime import datetime

pd.set_option("display.max_columns", 60)
pd.set_option("display.width", 180)

print("Entorno preparado.")

## BLOQUE 2 · Cargar los productos de Semana 7

In [ ]:
def cargar_archivo(nombre_esperado, contiene):
    try:
        from google.colab import files
        print(f"Selecciona el archivo {nombre_esperado} generado en Semana 7.")
        uploaded = files.upload()
        if not uploaded:
            raise ValueError("No se seleccionó ningún archivo.")
        ruta = Path(next(iter(uploaded)))
    except ImportError:
        ruta = Path(nombre_esperado)

    if not ruta.exists():
        raise FileNotFoundError(f"No se encontró: {ruta}")

    for enc in ["utf-8-sig", "utf-8", "latin-1"]:
        try:
            return pd.read_csv(ruta, encoding=enc), ruta
        except UnicodeDecodeError:
            pass
    raise UnicodeError("No fue posible leer el CSV con las codificaciones previstas.")

df_clima, RUTA_CLIMA_ORIGINAL = cargar_archivo("clima_validado_semana7.csv", "clima")
df_fito, RUTA_FITO_ORIGINAL = cargar_archivo("fitosanitario_validado_semana7.csv", "fitosanitario")

print("Clima:", RUTA_CLIMA_ORIGINAL.name, df_clima.shape)
print("Fitosanitario:", RUTA_FITO_ORIGINAL.name, df_fito.shape)
display(df_clima.head(3))
display(df_fito.head(3))

## BLOQUE 3 · Verificar columnas núcleo

In [ ]:
COLUMNAS_NUCLEO_CLIMA = [
    "fecha", "temperatura_oc", "temperatura_minima_oc", "temperatura_maxima_oc",
    "humedad_relativa", "humedad_relativa_minima", "humedad_relativa_maxima",
    "precipitacion_mm", "et0_mm", "velocidad_de_viento_promedio_km_h",
    "radiacion_solar_w_m2", "grados_dia_oc", "grados_dia_acumulado_ano_oc",
    "flag_temperatura_imposible", "flag_humedad_imposible", "estado_calidad",
]

COLUMNAS_NUCLEO_FITO = [
    "fecha", "campana", "modulo", "turno", "valvula", "plantas_evaluadas",
    "aranita_hojas_infestadas", "aranita_grado_hoja",
    "aranita_infestacion_postura", "aranita_grado_postura",
    "flag_duplicado_logico_corregido", "flag_pct_hoja_imposible", "flag_pct_postura_imposible",
    "flag_grado_hoja_atipico", "flag_grado_postura_atipico", "estado_calidad",
]

for nombre, df_chk, cols in [("clima", df_clima, COLUMNAS_NUCLEO_CLIMA), ("fitosanitario", df_fito, COLUMNAS_NUCLEO_FITO)]:
    faltantes = [c for c in cols if c not in df_chk.columns]
    if faltantes:
        raise ValueError(f"El archivo de {nombre} no cumple el esquema esperado. Faltan columnas: {faltantes}")
    print(f"OK: estructura mínima de {nombre} disponible.")

## BLOQUE 4 · Crear repositorio

In [ ]:
RAIZ = Path("/content/ARANITA_ROJA_DB")
if not RAIZ.exists():
    try:
        RAIZ = Path("/content/ARANITA_ROJA_DB")
        RAIZ.parent.mkdir(parents=True, exist_ok=True)
    except OSError:
        RAIZ = Path("./ARANITA_ROJA_DB")

if RAIZ.exists():
    shutil.rmtree(RAIZ)

CARPETAS = {
    "entrada": RAIZ / "01_entrada",
    "base": RAIZ / "02_base_datos",
    "documentacion": RAIZ / "03_documentacion",
    "exportaciones": RAIZ / "04_exportaciones",
    "respaldos": RAIZ / "05_respaldos",
}

for ruta in CARPETAS.values():
    ruta.mkdir(parents=True, exist_ok=True)

for nombre, ruta in CARPETAS.items():
    print(f"{nombre:15} -> {ruta}")

## BLOQUE 5 · Conservar entradas y establecer versión

In [ ]:
VERSION = "v1"
FECHA_VERSION = datetime.now().strftime("%Y-%m-%d")

RUTA_CLIMA = CARPETAS["entrada"] / f"clima_validado_{VERSION}_{FECHA_VERSION}.csv"
RUTA_FITO = CARPETAS["entrada"] / f"fitosanitario_validado_{VERSION}_{FECHA_VERSION}.csv"

df_clima.to_csv(RUTA_CLIMA, index=False, encoding="utf-8-sig")
df_fito.to_csv(RUTA_FITO, index=False, encoding="utf-8-sig")

print("Entradas conservadas:")
print(" -", RUTA_CLIMA.name)
print(" -", RUTA_FITO.name)

## BLOQUE 6 · Explorar cardinalidad de entidades de referencia

In [ ]:
for variable in ["modulo", "turno", "campana"]:
    print(f"{variable:10} -> {df_fito[variable].nunique(dropna=True)} valores distintos: {sorted(df_fito[variable].unique(), key=str)}")

## BLOQUE 7 · Crear tablas de dimensión

In [ ]:
def crear_dimension(dataframe, columna, nombre_id):
    dimension = (
        dataframe[[columna]].dropna().drop_duplicates().sort_values(columna).reset_index(drop=True)
    )
    dimension.insert(0, nombre_id, range(1, len(dimension) + 1))
    return dimension

dim_modulo = crear_dimension(df_fito, "modulo", "modulo_id")
dim_turno = crear_dimension(df_fito, "turno", "turno_id")
dim_campana = crear_dimension(df_fito, "campana", "campana_id")

display(dim_modulo)
display(dim_turno)
display(dim_campana)

## BLOQUE 8 · Construir CLIMA_DIARIO

In [ ]:
clima_diario = df_clima[COLUMNAS_NUCLEO_CLIMA].copy()
clima_diario = clima_diario.rename(columns={"estado_calidad": "estado_calidad_clima"})

for col in ["flag_temperatura_imposible", "flag_humedad_imposible"]:
    clima_diario[col] = clima_diario[col].astype(int)

fecha_no_repetida = ~clima_diario["fecha"].duplicated()
assert fecha_no_repetida.all(), "clima_diario no debe tener fechas repetidas (grano: 1 fila por día)."

print("CLIMA_DIARIO:", clima_diario.shape)
display(clima_diario.head())

## BLOQUE 9 · Construir EVALUACION_FITOSANITARIA con claves foráneas

In [ ]:
evaluacion = df_fito[COLUMNAS_NUCLEO_FITO].copy()
evaluacion = evaluacion.rename(columns={"estado_calidad": "estado_calidad_evaluacion"})

evaluacion = evaluacion.merge(dim_modulo, on="modulo", how="left")
evaluacion = evaluacion.merge(dim_turno, on="turno", how="left")
evaluacion = evaluacion.merge(dim_campana, on="campana", how="left")

for c in ["modulo_id", "turno_id", "campana_id"]:
    if evaluacion[c].isna().any():
        raise ValueError(f"Se generaron claves sin correspondencia en {c}.")
    evaluacion[c] = evaluacion[c].astype(int)

evaluacion = evaluacion[[
    "fecha", "modulo_id", "turno_id", "campana_id", "valvula", "plantas_evaluadas",
    "aranita_hojas_infestadas", "aranita_grado_hoja",
    "aranita_infestacion_postura", "aranita_grado_postura",
    "flag_duplicado_logico_corregido", "flag_pct_hoja_imposible", "flag_pct_postura_imposible",
    "flag_grado_hoja_atipico", "flag_grado_postura_atipico", "estado_calidad_evaluacion",
]].copy()

for col in ["flag_duplicado_logico_corregido", "flag_pct_hoja_imposible", "flag_pct_postura_imposible",
            "flag_grado_hoja_atipico", "flag_grado_postura_atipico"]:
    evaluacion[col] = evaluacion[col].astype(int)

print("Entrada        :", len(df_fito))
print("EVALUACIÓN      :", len(evaluacion))
assert len(df_fito) == len(evaluacion), "La transformación cambió el número de evaluaciones."

fechas_sin_clima = set(evaluacion["fecha"]) - set(clima_diario["fecha"])
if fechas_sin_clima:
    raise ValueError(f"Hay fechas de evaluación sin clima correspondiente: {sorted(fechas_sin_clima)[:5]} ...")

display(evaluacion.head())

## BLOQUE 10 · Crear base SQLite y esquema relacional

In [ ]:
RUTA_BD = CARPETAS["base"] / f"aranita_roja_bd_{VERSION}_{FECHA_VERSION}.sqlite"

conn = sqlite3.connect(RUTA_BD)
conn.execute("PRAGMA foreign_keys = ON;")
cursor = conn.cursor()

cursor.executescript("""
DROP TABLE IF EXISTS evaluacion_fitosanitaria;
DROP TABLE IF EXISTS clima_diario;
DROP TABLE IF EXISTS dim_modulo;
DROP TABLE IF EXISTS dim_turno;
DROP TABLE IF EXISTS dim_campana;

CREATE TABLE dim_modulo (
    modulo_id INTEGER PRIMARY KEY,
    modulo INTEGER NOT NULL UNIQUE
);

CREATE TABLE dim_turno (
    turno_id INTEGER PRIMARY KEY,
    turno INTEGER NOT NULL UNIQUE
);

CREATE TABLE dim_campana (
    campana_id INTEGER PRIMARY KEY,
    campana TEXT NOT NULL UNIQUE
);

CREATE TABLE clima_diario (
    fecha TEXT PRIMARY KEY,
    temperatura_oc REAL,
    temperatura_minima_oc REAL,
    temperatura_maxima_oc REAL,
    humedad_relativa REAL,
    humedad_relativa_minima REAL,
    humedad_relativa_maxima REAL,
    precipitacion_mm REAL,
    et0_mm REAL,
    velocidad_de_viento_promedio_km_h REAL,
    radiacion_solar_w_m2 REAL,
    grados_dia_oc REAL,
    grados_dia_acumulado_ano_oc REAL,
    flag_temperatura_imposible INTEGER,
    flag_humedad_imposible INTEGER,
    estado_calidad_clima TEXT
);

CREATE TABLE evaluacion_fitosanitaria (
    evaluacion_id INTEGER PRIMARY KEY AUTOINCREMENT,
    fecha TEXT NOT NULL,
    modulo_id INTEGER NOT NULL,
    turno_id INTEGER NOT NULL,
    campana_id INTEGER NOT NULL,
    valvula INTEGER NOT NULL,
    plantas_evaluadas INTEGER,
    aranita_hojas_infestadas REAL,
    aranita_grado_hoja REAL,
    aranita_infestacion_postura REAL,
    aranita_grado_postura REAL,
    flag_duplicado_logico_corregido INTEGER,
    flag_pct_hoja_imposible INTEGER,
    flag_pct_postura_imposible INTEGER,
    flag_grado_hoja_atipico INTEGER,
    flag_grado_postura_atipico INTEGER,
    estado_calidad_evaluacion TEXT,
    UNIQUE (fecha, modulo_id, valvula, turno_id),
    FOREIGN KEY (fecha) REFERENCES clima_diario(fecha),
    FOREIGN KEY (modulo_id) REFERENCES dim_modulo(modulo_id),
    FOREIGN KEY (turno_id) REFERENCES dim_turno(turno_id),
    FOREIGN KEY (campana_id) REFERENCES dim_campana(campana_id)
);
""")

conn.commit()
print("Esquema relacional creado.")

## BLOQUE 11 · Insertar tablas padre y tablas de hechos

In [ ]:
dim_modulo.to_sql("dim_modulo", conn, if_exists="append", index=False)
dim_turno.to_sql("dim_turno", conn, if_exists="append", index=False)
dim_campana.to_sql("dim_campana", conn, if_exists="append", index=False)
clima_diario.to_sql("clima_diario", conn, if_exists="append", index=False)
evaluacion.to_sql("evaluacion_fitosanitaria", conn, if_exists="append", index=False)

conn.commit()
print("Datos almacenados.")

## BLOQUE 12 · Verificar tablas y cantidad de registros

In [ ]:
tablas = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name;", conn
)
display(tablas)

nombres_tablas = ["dim_modulo", "dim_turno", "dim_campana", "clima_diario", "evaluacion_fitosanitaria"]

resumen = []
for tabla in nombres_tablas:
    n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {tabla};", conn).loc[0, "n"]
    resumen.append({"tabla": tabla, "registros": int(n)})

resumen_tablas = pd.DataFrame(resumen)
display(resumen_tablas)

## BLOQUE 13 · Comprobar integridad referencial

In [ ]:
integridad = pd.read_sql_query("PRAGMA foreign_key_check;", conn)

if integridad.empty:
    print("OK: no se detectaron violaciones de claves foráneas.")
else:
    print("ATENCIÓN: existen violaciones de integridad referencial.")
    display(integridad)

## BLOQUE 14 · Recuperar información mediante JOIN

In [ ]:
consulta = """
SELECT
    e.fecha,
    m.modulo,
    e.valvula,
    t.turno,
    c.campana,
    e.aranita_hojas_infestadas,
    e.aranita_grado_postura,
    cl.temperatura_oc,
    cl.grados_dia_oc,
    cl.grados_dia_acumulado_ano_oc
FROM evaluacion_fitosanitaria AS e
LEFT JOIN dim_modulo AS m ON e.modulo_id = m.modulo_id
LEFT JOIN dim_turno AS t ON e.turno_id = t.turno_id
LEFT JOIN dim_campana AS c ON e.campana_id = c.campana_id
LEFT JOIN clima_diario AS cl ON e.fecha = cl.fecha
LIMIT 10;
"""

resultado = pd.read_sql_query(consulta, conn)
display(resultado)

## BLOQUE 15 · Crear exportación controlada
Esta es la tabla base sobre la que se construirá el semáforo (Fase siguiente del proyecto): cada evaluación de campo ya con su clima y GDA del cultivo del mismo día.

In [ ]:
consulta_completa = """
SELECT
    e.fecha,
    m.modulo,
    e.valvula,
    t.turno,
    c.campana,
    e.plantas_evaluadas,
    e.aranita_hojas_infestadas,
    e.aranita_grado_hoja,
    e.aranita_infestacion_postura,
    e.aranita_grado_postura,
    e.flag_duplicado_logico_corregido,
    e.flag_grado_postura_atipico,
    cl.temperatura_oc,
    cl.temperatura_minima_oc,
    cl.temperatura_maxima_oc,
    cl.humedad_relativa,
    cl.grados_dia_oc,
    cl.grados_dia_acumulado_ano_oc
FROM evaluacion_fitosanitaria AS e
LEFT JOIN dim_modulo AS m ON e.modulo_id = m.modulo_id
LEFT JOIN dim_turno AS t ON e.turno_id = t.turno_id
LEFT JOIN dim_campana AS c ON e.campana_id = c.campana_id
LEFT JOIN clima_diario AS cl ON e.fecha = cl.fecha;
"""

df_exportado = pd.read_sql_query(consulta_completa, conn)

RUTA_EXPORTACION = CARPETAS["exportaciones"] / f"aranita_roja_exportacion_{VERSION}_{FECHA_VERSION}.csv"
df_exportado.to_csv(RUTA_EXPORTACION, index=False, encoding="utf-8-sig")

print("Exportación:", RUTA_EXPORTACION.name)
print("Filas:", len(df_exportado))

## BLOQUE 16 · Generar diccionario de datos desde SQLite

In [ ]:
descripciones = {
    "fecha": "Fecha del registro (clima diario o evaluación de campo)",
    "modulo_id": "Identificador del módulo de riego", "modulo": "Número de módulo del fundo",
    "turno_id": "Identificador del turno", "turno": "Turno de riego/evaluación",
    "campana_id": "Identificador de campaña agrícola", "campana": "Campaña agrícola (ej. 2025-2026)",
    "valvula": "Número de válvula dentro del módulo (no es único fuera de su módulo)",
    "plantas_evaluadas": "Cantidad de plantas evaluadas en el monitoreo",
    "aranita_hojas_infestadas": "% de hojas infestadas por arañita roja",
    "aranita_grado_hoja": "Índice de severidad en hoja (escala 0-4)",
    "aranita_infestacion_postura": "% de infestación de postura (huevos)",
    "aranita_grado_postura": "Índice de severidad en postura (escala 0-4)",
    "temperatura_oc": "Temperatura media diaria (°C)",
    "temperatura_minima_oc": "Temperatura mínima diaria (°C)",
    "temperatura_maxima_oc": "Temperatura máxima diaria (°C)",
    "humedad_relativa": "Humedad relativa media diaria (%)",
    "grados_dia_oc": "Grados-día del cultivo, calculado por la estación (no es el GDA de la plaga)",
    "grados_dia_acumulado_ano_oc": "Grados-día del cultivo acumulados en el año",
    "estado_calidad_clima": "Resultado de la validación de Semana 7 para ese día de clima",
    "estado_calidad_evaluacion": "Resultado de la validación de Semana 7 para esa evaluación",
}

filas_diccionario = []
for tabla in nombres_tablas:
    info = pd.read_sql_query(f"PRAGMA table_info({tabla});", conn)
    for _, fila in info.iterrows():
        campo = fila["name"]
        rol = "PK" if fila["pk"] == 1 else "Atributo"
        referencia = ""
        if tabla == "evaluacion_fitosanitaria" and campo in {"modulo_id", "turno_id", "campana_id"}:
            rol = "FK"
            referencia = {"modulo_id": "dim_modulo.modulo_id", "turno_id": "dim_turno.turno_id",
                          "campana_id": "dim_campana.campana_id"}[campo]
        if tabla == "evaluacion_fitosanitaria" and campo == "fecha":
            rol = "FK"
            referencia = "clima_diario.fecha"
        filas_diccionario.append({
            "tabla": tabla, "campo": campo, "tipo": fila["type"], "rol": rol,
            "obligatorio": "Sí" if fila["notnull"] == 1 else "No",
            "descripcion": descripciones.get(campo, ""), "referencia": referencia,
        })

diccionario = pd.DataFrame(filas_diccionario)
display(diccionario)

RUTA_DICCIONARIO = CARPETAS["documentacion"] / f"diccionario_datos_{VERSION}_{FECHA_VERSION}.csv"
diccionario.to_csv(RUTA_DICCIONARIO, index=False, encoding="utf-8-sig")

## BLOQUE 17 · Generar manifiesto de versión

In [ ]:
manifiesto = {
    "proyecto": "Sistema de alerta temprana — arañita roja, Fundo Loma Verde S.A.C.",
    "semana": 8,
    "actividad": "Organización y almacenamiento",
    "version": VERSION,
    "fecha_generacion": datetime.now().isoformat(timespec="seconds"),
    "archivo_fuente_clima": RUTA_CLIMA.name,
    "archivo_fuente_fitosanitario": RUTA_FITO.name,
    "base_datos": RUTA_BD.name,
    "motor": "SQLite",
    "tablas": nombres_tablas,
    "dias_clima": int(len(clima_diario)),
    "evaluaciones_fitosanitarias": int(len(evaluacion)),
    "diccionario_datos": RUTA_DICCIONARIO.name,
    "exportacion": RUTA_EXPORTACION.name,
    "pendiente_siguiente_fase": "Calcular GDA de la plaga (Tbase=10.34°C, K=163.93 GDA) y construir el semáforo de alerta.",
}

RUTA_MANIFIESTO = CARPETAS["documentacion"] / f"manifiesto_{VERSION}_{FECHA_VERSION}.json"
with open(RUTA_MANIFIESTO, "w", encoding="utf-8") as f:
    json.dump(manifiesto, f, ensure_ascii=False, indent=4)

print(json.dumps(manifiesto, ensure_ascii=False, indent=4))

## BLOQUE 18 · Cerrar base y mostrar repositorio

In [ ]:
conn.commit()
conn.close()

for raiz, carpetas, archivos in os.walk(RAIZ):
    nivel = raiz.replace(str(RAIZ), "").count(os.sep)
    sangria = "    " * nivel
    print(f"{sangria}{Path(raiz).name}/")
    for archivo in archivos:
        print(f"{sangria}    {archivo}")

## BLOQUE 19 · Crear respaldo

In [ ]:
RUTA_ZIP_TEMP = RAIZ.parent / f"ARANITA_ROJA_respaldo_{VERSION}_{FECHA_VERSION}"

archivo_zip = shutil.make_archive(str(RUTA_ZIP_TEMP), "zip", root_dir=RAIZ)

DESTINO_RESPALDO = CARPETAS["respaldos"] / f"ARANITA_ROJA_respaldo_{VERSION}_{FECHA_VERSION}.zip"
shutil.move(archivo_zip, DESTINO_RESPALDO)

print("Respaldo:", DESTINO_RESPALDO)
print("Existe:", DESTINO_RESPALDO.exists())
print("Tamaño KB:", round(DESTINO_RESPALDO.stat().st_size / 1024, 2))

## BLOQUE 20 · Simular recuperación del respaldo

In [ ]:
RUTA_RECUPERACION = RAIZ.parent / "ARANITA_ROJA_RECUPERADO"

if RUTA_RECUPERACION.exists():
    shutil.rmtree(RUTA_RECUPERACION)
RUTA_RECUPERACION.mkdir(parents=True)

shutil.unpack_archive(str(DESTINO_RESPALDO), str(RUTA_RECUPERACION))

bases_recuperadas = list(RUTA_RECUPERACION.rglob("*.sqlite"))
if not bases_recuperadas:
    raise FileNotFoundError("El respaldo se extrajo, pero no contiene una base SQLite.")

BD_RECUPERADA = bases_recuperadas[0]
print("Base recuperada:", BD_RECUPERADA)

## BLOQUE 21 · Verificar que la base recuperada funciona

In [ ]:
conn_rec = sqlite3.connect(BD_RECUPERADA)
conn_rec.execute("PRAGMA foreign_keys = ON;")

conteo = pd.read_sql_query("SELECT COUNT(*) AS registros FROM evaluacion_fitosanitaria;", conn_rec)
n_original = len(evaluacion)
n_recuperado = int(conteo.loc[0, "registros"])

integridad_rec = pd.read_sql_query("PRAGMA foreign_key_check;", conn_rec)
conn_rec.close()

print("Evaluaciones originales  :", n_original)
print("Evaluaciones recuperadas :", n_recuperado)
print("Integridad FK            :", "OK" if integridad_rec.empty else "REVISAR")

if n_original == n_recuperado and integridad_rec.empty:
    print("\nRECUPERACIÓN VERIFICADA.")
else:
    print("\nREVISAR RESPALDO O BASE RECUPERADA.")

## BLOQUE 22 · Reporte final

In [ ]:
estado_recuperacion = "VERIFICADA" if n_original == n_recuperado and integridad_rec.empty else "REVISAR"

print("=" * 65)
print("SISTEMA DE ALERTA TEMPRANA — ARAÑITA ROJA · SEMANA 8")
print("ORGANIZACIÓN Y ALMACENAMIENTO")
print("=" * 65)

print(f"""
Días de clima almacenados:
{len(clima_diario)}

Evaluaciones fitosanitarias almacenadas:
{len(evaluacion)}

Tablas:
{len(nombres_tablas)}

Base de datos:
{RUTA_BD.name}

Diccionario:
{RUTA_DICCIONARIO.name}

Manifiesto:
{RUTA_MANIFIESTO.name}

Exportación:
{RUTA_EXPORTACION.name}

Respaldo:
{DESTINO_RESPALDO.name}

Recuperación:
{estado_recuperacion}
""")

## Producto final de Semana 8
El repositorio contiene: entrada controlada, base SQLite con 2 tablas de hechos (`clima_diario`, `evaluacion_fitosanitaria`) y 3 dimensiones, diccionario de datos, manifiesto, exportación (la tabla base para el semáforo) y respaldo verificado.

**Pendiente para la siguiente fase:** calcular el GDA de la plaga sobre `clima_diario` (Tbase=10.34°C, K=163.93 GDA) y construir el semáforo de alerta sobre la exportación generada aquí.